# Build the ERCOT 20-zone GenX inputs

Run this notebook from `example_systems/pypsa_data_tools/ERCOT/`, using the Python
environment in [the build guide](README.md#python-environment).

1. Download the inputs and run the checks in Step 0.
2. Run Steps 1–10 in order to build the network and time series.
3. Run Step 11 to export the GenX CSVs and check them.
4. Run Step 12 to write the thermal-cap inputs.

The default output is `example_systems/13_ERCOT_20_zone/`. Set
`ERCOT20_CASE_BASELINE` before starting the notebook to change it.
ERA5 downloads and renewable profiles can take several hours.

This notebook provides the execution sequence and output paths. See the
[ERCOT README](README.md) for data preparation and the
[network README](network/README.md) for clustering, capacity and impedance methods.
Demand and weather use 2022, set by `scripts/era5_config.WEATHER_YEAR`.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Run from this directory, in the environment from ../requirements.txt.
ERCOT_REBUILD = Path.cwd()
assert ERCOT_REBUILD.name == "ERCOT", (
    "Run this notebook with working directory set to "
    "example_systems/pypsa_data_tools/ERCOT/"
)
sys.path.insert(0, str(ERCOT_REBUILD / "network"))
sys.path.insert(0, str(ERCOT_REBUILD / "scripts"))

def run(script_rel_path, cwd_rel, *args, env_extra=None):
    """Run a script in its working directory and print captured output."""
    cwd = ERCOT_REBUILD / cwd_rel
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    print(f"=== running {cwd_rel}/{script_rel_path} ===")
    result = subprocess.run(
        [sys.executable, script_rel_path, *map(str, args)], cwd=str(cwd), env=env,
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"{script_rel_path} failed (exit {result.returncode})")


## Step 0 — Source data

No source data is included in the repository. Get each input below and save it at the path
shown (relative to this directory). Then run the check cell, which lists anything still
missing.

| # | Input | Where to get it | Save as |
|---|---|---|---|
| 1 | Texas7k synthetic grid, 2022-09-23 case (`Texas7k_20220923.RAW`, `.AUX`) | TAMU Electric Grid Test Cases, https://electricgrids.engr.tamu.edu | any folder; set `$TEXAS7K_RAW_DIR` to it. The conversion cell below writes `Texas7k_2022_pypsausa_format/*.csv` |
| 2 | ERCOT hourly native load, 2022 | ERCOT, https://www.ercot.com/gridinfo/load/load_hist | `source_data/ERCOT_Native_Load/Native_Load_2022.xlsx` |
| 3 | NLR county-level hourly load, 2016–2023 | https://data.openei.org/submissions/8562 (Obika, Cole & Rivers, *Sci Data* 2026, https://doi.org/10.1038/s41597-026-07693-3). Download the national file and prepare the Texas subset as described in [the build guide](README.md#data-sources) | `source_data/NREL_Nature_Demand/historic_load_hourly_2016_2023_texas.h5` |
| 4 | ERA5 weather cutout, 2022 (~5 GB) | Copernicus Climate Data Store. Needs a free account and `~/.cdsapirc` (https://cds.climate.copernicus.eu/how-to-api). The ERA5 cell below downloads it | `source_data/usa_era5_2022.nc` |
| 5 | EIA-860 generators | PUDL data release, Catalyst Cooperative, https://data.catalyst.coop | `source_data/out_eia__yearly_generators.parquet` |
| 6 | NLR ATB cost and financial tables | Same PUDL release | `source_data/core_nrelatb__yearly_projected_cost_performance.parquet`, `source_data/core_nrelatb__yearly_projected_financial_cases_by_scenario.parquet` |
| 7 | NLR ATB 2022 v2 workbook | https://data.openei.org/submissions/5716 (2022 v2) | `source_data/2022 v2 Annual Technology Baseline Workbook Corrected 7-21-2022.xlsx` |
| 8 | HIFLD Electric Power Transmission Lines (GeoJSON) | [HIFLD archive](https://www.arcgis.com/home/item.html?id=d4090758322c4d32a4cd002ffaa0aa12); export as GeoJSON. The national file works as is; clipping to Texas only reduces its size | `source_data/HIFLD/electric_transmission_lines_tx_bbox.geojson` |
| 9 | Land-cover and protected-area rasters, US county shapes | PyPSA-USA data bundle, https://zenodo.org/records/14219029 (`pypsa_usa_data.zip`). Extract so the listed files are directly under the destination folder | `source_data/pypsa_usa_data/` (`copernicus/…tif`, `natura.tiff`, `counties/cb_2020_us_county_500k.*`) |
| 10 | PyPSA-USA `repo_data` (ReEDS and balancing-area shapes) | Sparse clone of https://github.com/PyPSA/pypsa-usa at commit `f38d756` (commands in `network/paths.py`) | `source_data/pypsa_usa_repo_data/` |
| 11 | EIA fuel prices (Texas gas monthly, coal quarterly) | EIA Open Data API, fetched automatically in Step 10. Needs a free key (https://www.eia.gov/opendata/register.php) in `$EIA_API_KEY` | `source_data/EIA/` (written by Step 10) |

The versions and download dates used for the case are listed in [the build guide](README.md#data-sources). PUDL,
HIFLD and the EIA prices are updated over time, so newer downloads can give slightly
different inputs.


In [ ]:
from paths import SOURCE_DATA, PYPSA_USA_DATA, TEXAS7K_CSV_DIR, REEDS_SHAPES, PYPSA_USA_REPO_DATA
from era5_config import WEATHER_YEAR

PUDL = "PUDL data release (https://data.catalyst.coop), Parquet files"
PYPSA_USA_BUNDLE = "PyPSA-USA data bundle (https://zenodo.org/records/14219029, extract pypsa_usa_data.zip under source_data/pypsa_usa_data/)"
SOURCES = [
    (TEXAS7K_CSV_DIR / "buses.csv", "Texas7k network CSVs: run the conversion cell below"),
    (SOURCE_DATA / "ERCOT_Native_Load" / f"Native_Load_{WEATHER_YEAR}.xlsx",
     "ERCOT hourly native load, https://www.ercot.com/gridinfo/load/load_hist"),
    (SOURCE_DATA / "NREL_Nature_Demand" / "historic_load_hourly_2016_2023_texas.h5",
     "NLR county hourly load, https://data.openei.org/submissions/8562; prepare the Texas subset using README.md, Data sources"),
    (SOURCE_DATA / f"usa_era5_{WEATHER_YEAR}.nc", "ERA5 cutout: run the ERA5 cell below"),
    (SOURCE_DATA / "out_eia__yearly_generators.parquet", PUDL),
    (SOURCE_DATA / "core_nrelatb__yearly_projected_cost_performance.parquet", PUDL),
    (SOURCE_DATA / "core_nrelatb__yearly_projected_financial_cases_by_scenario.parquet", PUDL),
    (SOURCE_DATA / "2022 v2 Annual Technology Baseline Workbook Corrected 7-21-2022.xlsx",
     "NLR ATB 2022 v2 workbook, https://data.openei.org/submissions/5716"),
    (SOURCE_DATA / "HIFLD" / "electric_transmission_lines_tx_bbox.geojson",
     "HIFLD archive (https://www.arcgis.com/home/item.html?id=d4090758322c4d32a4cd002ffaa0aa12) as GeoJSON. "
     "The national file can be used as is; we clipped it to Texas only to keep the file small"),
    (PYPSA_USA_DATA / "copernicus" /
     "PROBAV_LC100_global_v3.0.1_2019-nrt_Discrete-Classification-map_USA_EPSG-4326.tif", PYPSA_USA_BUNDLE),
    (PYPSA_USA_DATA / "natura.tiff", PYPSA_USA_BUNDLE),
    (PYPSA_USA_DATA / "counties" / "cb_2020_us_county_500k.shp", PYPSA_USA_BUNDLE),
    (REEDS_SHAPES, "PyPSA-USA repo_data: sparse-clone into source_data/pypsa_usa_repo_data/ "
                   "(commands in network/paths.py)"),
    (PYPSA_USA_REPO_DATA / "geospatial" / "BA_shapes_new" / "Modified_BE_BA_Shapes.shp",
     "PyPSA-USA repo_data (as above)"),
]

missing = 0
for path, how in SOURCES:
    found = path is not None and Path(path).exists()
    missing += not found
    shown = Path(path).relative_to(ERCOT_REBUILD) if path and Path(path).is_relative_to(ERCOT_REBUILD) else path
    print(f"{'ok     ' if found else 'MISSING'}  {shown}")
    if not found:
        print(f"         -> {how}")
print(f"\n{len(SOURCES) - missing}/{len(SOURCES)} inputs found")
if not os.environ.get("EIA_API_KEY") and not (SOURCE_DATA / "EIA" / "gas_price_tx_monthly.csv").exists():
    print("EIA_API_KEY is not set and no cached EIA prices were found; Step 10 will need it.")

In [ ]:
# Texas7k RAW/AUX -> PyPSA-format CSVs (only needed once)
if not (TEXAS7K_CSV_DIR / "buses.csv").exists():
    run("convert_texas7k_2022_to_pypsausa.py", "Texas7k_2022_pypsausa_format",
        env_extra={"TEXAS7K_RAW_DIR": os.environ.get("TEXAS7K_RAW_DIR", str(TEXAS7K_CSV_DIR))})

In [ ]:
# ERA5 weather cutout (only needed once; ~5 GB download, can take hours)
if not (SOURCE_DATA / f"usa_era5_{WEATHER_YEAR}.nc").exists():
    run("build_era5_cutout.py", "scripts", "--year", WEATHER_YEAR)

## Step 1 — Load the ERCOT network

Load the converted Texas7k CSVs and filter buses to ERCOT with
`network/build_and_cluster_texas7k.py`.

Outputs: `intermediates/texas7k_raw_network.nc` and
`intermediates/texas7k_buses_with_reeds_zone.csv`.


In [ ]:
run("build_and_cluster_texas7k.py", "network")

### Optional: connectivity check

Run `network/check_connectivity.py` to report connected components before clustering.


In [ ]:
run("check_connectivity.py", "network")

## Step 2 — Build clustering features

Run both cells before Step 3:

- `network/build_hifld_line_confirmation.py`: HIFLD line-match features.
- `scripts/build_bus_vre_features_for_hac.py`: per-bus wind and solar profiles.

Requires the ERA5 cutout from Step 0. See [clustering methods](network/README.md#clustering).


In [ ]:
run("build_hifld_line_confirmation.py", "network")

In [ ]:
run("build_bus_vre_features_for_hac.py", "scripts")

## Step 3 — Cluster to 20 zones

Run `network/cluster_texas7k.py` using the features from Step 2.

Outputs: `intermediates/texas7k_clustered_20.nc` and
`intermediates/texas7k_busmap_20zone.csv`. Subsequent steps reuse this zone ordering.


In [ ]:
run("cluster_texas7k.py", "network")

### Check inter-zone corridors

Run `network/verify_345_lines.py` to filter corridors against HIFLD.
It updates `intermediates/texas7k_clustered_20.nc` in place.


In [ ]:
run("verify_345_lines.py", "network")

## Step 4 — Calculate line capacities

Run `network/lpe_all_lines.py`.
Output: `outputs/lpe_all_lines_capacity_20zone.csv`, including the limiting branch
used in Step 11. See [capacity methods](network/README.md#line-capacity-lpe).


In [ ]:
run("lpe_all_lines.py", "network")

## Step 5 — Calculate line impedances

Run `network/kron_reduce_sparse.py`.
Output: `outputs/kron_reduced_capacity_20zone.csv`.
See [impedance methods](network/README.md#line-impedance).


In [ ]:
run("kron_reduce_sparse.py", "network")

### Write the network with fallback impedance

Run `network/apply_conductor_impedance.py` even when the Kron output covers every
zone pair. It writes `intermediates/texas7k_clustered_20_realimpedance.nc`, which
Step 11 reads. Conductor-based values provide fallback impedance for missing pairs.


In [ ]:
run("apply_conductor_impedance.py", "network")

## Step 6 — Build demand

Run `scripts/build_demand_nrel_ercot_wz.py` after the network steps.
Output: `outputs/ercot_demand_20zone_nrel_ercot_wz.csv`.
Step 7 uses its timestamps; Step 11 exports it to GenX.
See [demand inputs and allocation](README.md#demand).


In [ ]:
run("build_demand_nrel_ercot_wz.py", "scripts")


## Step 7 — Build wind and solar profiles

Run both cells in order:

- `scripts/build_bus_regions.py` → `outputs/ercot_zone_regions.geojson`.
- `scripts/build_vre_profiles.py` →
  `outputs/ercot_{onwind,solar}_{profile,potential}_20zone.csv`.

Requires the ERA5 cutout, geographic inputs from Step 0 and demand from Step 6.


In [ ]:
run("build_bus_regions.py", "scripts")

In [ ]:
run("build_vre_profiles.py", "scripts")

## Step 8 — Prepare generator data

- `scripts/build_generators.py` → `outputs/ercot_generators_pudl_existing.csv`.
- `scripts/map_generators_to_buses.py` → `outputs/ercot_generators_mapped.csv`.

Step 11 reads the mapped generator data when writing thermal resource inputs.


In [ ]:
run("build_generators.py", "scripts")

In [ ]:
run("map_generators_to_buses.py", "scripts")

## Step 9 — Prepare technology costs

Run `scripts/build_tech_costs.py` using the PUDL ATB tables and ATB workbook from Step 0.
Output: `outputs/atb_tech_costs_2022.csv`.


In [ ]:
run("build_tech_costs.py", "scripts")

## Step 10 — Prepare fuel prices

Run `source_data/EIA/build_fuel_prices.py` to fetch Texas gas and coal prices from EIA
and write the time series to `source_data/EIA/`.
Set `EIA_API_KEY` for the initial download; subsequent runs can use cached responses.


In [ ]:
run("build_fuel_prices.py", "source_data/EIA")

## Step 11 — Export GenX inputs

Run each cell below in order. They assemble the preceding outputs into `system/` and
`resources/` under `ERCOT20_CASE_BASELINE` (default: `example_systems/13_ERCOT_20_zone/`).


In [ ]:
import argparse

import pandas as pd
import numpy as np
import pypsa

from era5_config import WEATHER_YEAR
from network_paths import (
    CLUSTERED_REALIMPEDANCE_NC,
    LPE_ALL_LINES_CSV,
    KRON_CAPACITY_CSV,
    OUTPUTS,
    SOURCE_DATA,
    SINGLE_CIRCUIT_MVA,
    require,
)
from paths import CASE_BASELINE

# Align the fuel-price time index with the weather year.
FUEL_YEAR = WEATHER_YEAR

CASE = {"costs": "atb_tech_costs_2022.csv", "out": str(CASE_BASELINE)}

RAW_OUT = str(OUTPUTS)
RAW_SRC = str(SOURCE_DATA)
RAW = RAW_OUT  # outputs of Steps 1-10
OUT = CASE["out"]
print(f"Writing GenX case to {OUT}")

os.makedirs(f"{OUT}/system", exist_ok=True)
os.makedirs(f"{OUT}/resources", exist_ok=True)

n = pypsa.Network(str(require(CLUSTERED_REALIMPEDANCE_NC, "clustered network with conductor impedance")))
N_ZONES = len(n.buses)
zone_names = [f"z{i+1}" for i in range(N_ZONES)]

### Network.csv

Combine the clustered network, Step 4 capacities and Step 5 impedances.
The cells below add voltage limits, per-circuit ratings and `Line_Loss_Percentage`.


In [ ]:
zones_df = pd.DataFrame({
    "Network_zones": zone_names,
    "Zone_Voltage_Magnitude_Maximum": 1.05 * n.buses["v_nom"].to_numpy(),
    "Zone_Voltage_Magnitude_Minimum": 0.95 * n.buses["v_nom"].to_numpy(),
    "Zone_Shunt_Conductance_Siemens": 0,
    "Zone_Shunt_Susceptance_Siemens": 0,
    "Unnamed: 0": [f"ERCOT{i+1}" for i in range(N_ZONES)],
})

bus_to_zoneidx = {b: i + 1 for i, b in enumerate(n.buses.index)}
lines = n.lines.copy()
lines["Start_Zone"] = lines["bus0"].map(bus_to_zoneidx)
lines["End_Zone"] = lines["bus1"].map(bus_to_zoneidx)
lines["Line_Voltage_kV"] = 345.0

# Capacity from LPE, keyed on the zone pair.
lpe = pd.read_csv(require(LPE_ALL_LINES_CSV, "LPE line capacities"))
lpe_cap = {frozenset((int(r.zone_i), int(r.zone_j))): r.F_mw for r in lpe.itertuples()}
keys = [frozenset((int(a), int(b))) for a, b in zip(lines["Start_Zone"], lines["End_Zone"])]
missing = [k for k in keys if k not in lpe_cap]
if missing:
    raise RuntimeError(
        f"{len(missing)} clustered line(s) have no LPE capacity, e.g. {[set(k) for k in missing[:3]]}. "
        f"Re-run lpe_all_lines.py against this clustering."
    )
lines["Line_Max_Flow_MW"] = [lpe_cap[k] for k in keys]
lines["Line_Max_Reinforcement_MW"] = lines["Line_Max_Flow_MW"]
print(f"Line capacity from LPE: {len(lines)} lines, "
      f"{lines['Line_Max_Flow_MW'].sum():,.0f} MW total "
      f"(median {lines['Line_Max_Flow_MW'].median():,.0f} MW)")
lines["Network_Lines"] = np.arange(1, len(lines) + 1)
lines["transmission_path_name"] = lines["Start_Zone"].astype(str) + "_to_" + lines["End_Zone"].astype(str)
lines["distance_mile"] = lines["length"] * 0.621371

In [ ]:
# Scale LPE binding-branch ratings to 345 kV; use the Texas7k median as fallback.
V_KV_BASIS = 345.0
SINGLE_CIRCUIT_MVA_FALLBACK = SINGLE_CIRCUIT_MVA
if "binding_s_nom" in lpe.columns:
    lpe["single_line_345_equiv"] = lpe["binding_s_nom"] * (V_KV_BASIS / lpe["binding_v_nom"])
    single_line_map = {frozenset((int(r.zone_i), int(r.zone_j))): r.single_line_345_equiv
                       for r in lpe.itertuples()}
    lines["Single_Line_MW"] = [single_line_map.get(k, SINGLE_CIRCUIT_MVA_FALLBACK) for k in keys]
    print(f"Single_Line_MW: per-corridor, voltage-scaled from LPE's real binding branch "
          f"(median {lines['Single_Line_MW'].median():.0f} MVA, "
          f"fallback {SINGLE_CIRCUIT_MVA_FALLBACK:.0f} MVA for any unmatched pair)")
else:
    lines["Single_Line_MW"] = SINGLE_CIRCUIT_MVA_FALLBACK
    print("Single_Line_MW: lpe_all_lines_capacity_20zone.csv has no binding_s_nom column "
          f"(re-run lpe_all_lines.py to get it) -- using statewide fallback "
          f"{SINGLE_CIRCUIT_MVA_FALLBACK:.0f} MVA for all lines")

# Initial parallel-circuit count from capacity and per-circuit rating.
lines["Line_Num_Parallel"] = lines["Line_Max_Flow_MW"] / lines["Single_Line_MW"]

lines["Line_Loss_Percentage"] = 0.025  # See ../calibrate_losses.py.
# PyPSA-USA HVAC overhead cost (TEPCC 2023), annualized at 4.4% over 60 years.
ANNUALIZED_TX_COST_PER_MW_MILE = 118.03
lines["Line_Reinforcement_Cost_per_MWyr"] = ANNUALIZED_TX_COST_PER_MW_MILE * lines["length"] * 0.621371

In [ ]:
# Apply Kron impedance; the correction cell below adjusts the circuit count.
kron = pd.read_csv(str(require(KRON_CAPACITY_CSV, "Kron-reduced impedance"))) \
    if os.path.exists(str(KRON_CAPACITY_CSV)) else None
if kron is not None and {"r_eq_ohm", "x_eq_ohm"}.issubset(kron.columns):
    kron_map = {frozenset((int(r.zone_i), int(r.zone_j))): (r.r_eq_ohm, r.x_eq_ohm)
               for r in kron.itertuples()}
    missing_kron = [k for k in keys if k not in kron_map]
    if missing_kron:
        print(f"WARNING: {len(missing_kron)} line(s) have no Kron impedance "
              f"(e.g. {[set(k) for k in missing_kron[:3]]}) -- falling back to conductor-spec "
              f"r/x for those only.")
    n_per_pair = lines["Line_Num_Parallel"].to_numpy()
    r_eq_ohm = np.array([kron_map[k][0] if k in kron_map else None for k in keys], dtype=object)
    x_eq_ohm = np.array([kron_map[k][1] if k in kron_map else None for k in keys], dtype=object)
    fallback_mask = np.array([k not in kron_map for k in keys])
    lines["Line_Resistance_Ohms"] = [
        (r_eq_ohm[i] * n_per_pair[i]) if not fallback_mask[i] else lines["r"].iloc[i]
        for i in range(len(lines))
    ]
    lines["Line_Reactance_Ohms"] = [
        (x_eq_ohm[i] * n_per_pair[i]) if not fallback_mask[i] else lines["x"].iloc[i]
        for i in range(len(lines))
    ]
    print(f"Line_Reactance_Ohms: Kron-derived (de-combined to per-circuit via "
          f"Line_Num_Parallel) for {(~fallback_mask).sum()}/{len(lines)} lines "
          f"(median {lines['Line_Reactance_Ohms'].median():.2f} ohm); "
          f"{fallback_mask.sum()} fell back to conductor-spec r/x")
else:
    lines["Line_Resistance_Ohms"] = lines["r"]
    lines["Line_Reactance_Ohms"] = lines["x"]
    print("Line_Resistance_Ohms/Line_Reactance_Ohms: kron_reduced_capacity_20zone.csv not "
          "found or missing r_eq_ohm/x_eq_ohm columns -- using conductor-spec r/x "
          "(re-run kron_reduce_sparse.py to get the Kron-derived values)")
lines["Line_Charging_Susceptance_Siemens"] = lines["b"]
lines["Angle_Limit_Rad"] = np.pi / 6
lines["Tap_Ratios"] = 1
lines["Phase_Shift_Degrees"] = 0
lines["CapRes_Excl_1"] = 0
lines["DerateCapRes_1"] = 0.95

In [ ]:
zone_cols = ["Network_zones", "Zone_Voltage_Magnitude_Maximum", "Zone_Voltage_Magnitude_Minimum",
             "Zone_Shunt_Conductance_Siemens", "Zone_Shunt_Susceptance_Siemens", "Unnamed: 0"]
line_cols = ["Network_Lines", "Start_Zone", "End_Zone", "Line_Max_Flow_MW", "transmission_path_name",
             "distance_mile", "Line_Loss_Percentage", "Single_Line_MW", "Line_Num_Parallel",
             "Line_Max_Reinforcement_MW", "Line_Reinforcement_Cost_per_MWyr", "DerateCapRes_1",
             "CapRes_Excl_1", "Line_Resistance_Ohms", "Line_Reactance_Ohms",
             "Line_Charging_Susceptance_Siemens", "Line_Voltage_kV", "Angle_Limit_Rad",
             "Tap_Ratios", "Phase_Shift_Degrees"]
genx_cols = zone_cols + line_cols

n_over = min(len(zones_df), len(lines))
top = pd.concat([zones_df[zone_cols].iloc[:n_over].reset_index(drop=True),
                 lines[line_cols].iloc[:n_over].reset_index(drop=True)], axis=1)
if len(zones_df) > len(lines):
    bottom = pd.concat([zones_df[zone_cols].iloc[n_over:].reset_index(drop=True),
                         pd.DataFrame(np.nan, index=range(len(zones_df) - n_over), columns=line_cols)], axis=1)
elif len(lines) > len(zones_df):
    bottom = pd.concat([pd.DataFrame(np.nan, index=range(len(lines) - n_over), columns=zone_cols),
                         lines[line_cols].iloc[n_over:].reset_index(drop=True)], axis=1)
else:
    bottom = pd.DataFrame(columns=genx_cols)
network = pd.concat([top, bottom], ignore_index=True)[genx_cols]
network["Network_Lines"] = network["Network_Lines"].astype("Int64")
network["Start_Zone"] = network["Start_Zone"].astype("Int64")
network["End_Zone"] = network["End_Zone"].astype("Int64")
network.to_csv(f"{OUT}/system/Network.csv", index=False)
print(f"Network.csv: {len(network)} rows")
network.head()

### Apply per-circuit impedance correction

Run `network/correct_realN_impedance.py` on the exported network.
See [per-circuit impedance](network/README.md#impedance-per-circuit) for the conversion.


In [ ]:
run("correct_realN_impedance.py", "network", OUT)


### Demand_data.csv

Export Step 6 demand with the demand-curtailment settings defined below.


In [ ]:
# Read the Step 6 demand output.
sys.path.insert(0, "scripts")
from demand_source import demand_csv
_demand_path = demand_csv()
print(f"Demand: {_demand_path.name}")
demand = pd.read_csv(_demand_path)
# local_time is only used to align weather and demand; it is not a GenX column.
demand = demand.drop(columns=[c for c in ["local_time"] if c in demand.columns])
demand_additional = pd.DataFrame([], columns=["Voll", "Demand_Segment", "Cost_of_Demand_Curtailment_per_MW",
                                               "Max_Demand_Curtailment", "$/MWh", "Rep_Periods",
                                               "Timesteps_per_Rep_Period", "Sub_Weights"])
demand_additional.loc[0, "Voll"] = 5000
demand_additional.loc[0, "Demand_Segment"] = 1
demand_additional.loc[0, "Cost_of_Demand_Curtailment_per_MW"] = 1
demand_additional.loc[0, "Max_Demand_Curtailment"] = 1
demand_additional.loc[0, "$/MWh"] = 2000
demand_additional.loc[0, "Rep_Periods"] = 1
demand_additional.loc[0, "Timesteps_per_Rep_Period"] = len(demand)
demand_additional.loc[0, "Sub_Weights"] = len(demand)
demand_out = pd.concat([demand_additional, demand.rename(columns={c: f"Demand_MW_{c}" for c in demand.columns if c != "Time_Index"})], axis=1)
demand_out.to_csv(f"{OUT}/system/Demand_data.csv", index=False)
print(f"Demand_data.csv: {len(demand_out)} rows")


### Fuels_data.csv

Export Step 10 gas and coal prices.


In [ ]:
gas = pd.read_csv(f"{RAW_SRC}/EIA/gas_price_tx_monthly.csv")
gas["period"] = pd.to_datetime(gas["period"])
coal = pd.read_csv(f"{RAW_SRC}/EIA/coal_price_tx_quarterly.csv")

# Gas prices are monthly for the model year; broadcast each month's price to its hours.
if sorted(gas["period"].dt.year.unique()) != [FUEL_YEAR]:
    raise RuntimeError(
        f"gas_price_tx_monthly.csv covers {sorted(gas['period'].dt.year.unique())}, "
        f"expected only {FUEL_YEAR}. Re-run source_data/EIA/build_fuel_prices.py."
    )
hours = pd.date_range(f"{FUEL_YEAR}-01-01", periods=8760, freq="h")
gas_by_month = gas.set_index(gas["period"].dt.month)["usd_per_mmbtu"]
if len(gas_by_month) != 12:
    raise RuntimeError(f"expected 12 monthly gas prices, got {len(gas_by_month)}")
gas_hourly = hours.month.map(gas_by_month)

# Broadcast the mean coal price to all hours.
coal_avg = coal["usd_per_mmbtu"].mean()
coal_hourly = np.full(8760, coal_avg)

fuel_prices = pd.DataFrame({"Time_Index": np.arange(1, 8761), "Gas": gas_hourly, "coal": coal_hourly})
emissions_row = pd.DataFrame([{"Time_Index": 0, "Gas": 52.91 / 1000, "coal": 95.99333333333333 / 1000}])
fuel_prices_out = pd.concat([emissions_row, fuel_prices], ignore_index=True)
fuel_prices_out.to_csv(f"{OUT}/system/Fuels_data.csv", index=False)
print(f"Fuels_data.csv: {len(fuel_prices_out)} rows, real EIA monthly gas ({gas_hourly.min():.2f}-{gas_hourly.max():.2f} $/MMBtu) + quarterly-avg coal ({coal_avg:.2f} $/MMBtu)")

### Thermal.csv

Combine Step 8 generator locations and Step 9 costs into thermal resource rows.


In [ ]:
atb = pd.read_csv(f"{RAW}/{CASE['costs']}", index_col=0)

real_gens = pd.read_csv(f"{RAW}/ercot_generators_mapped.csv")
coal_buses = set(real_gens.loc[real_gens["fuel_type_code_pudl"] == "coal", "bus"].unique())
bus_to_zoneidx_str = {str(b): i + 1 for i, b in enumerate(n.buses.index)}
coal_zones = {bus_to_zoneidx_str[str(b)] for b in coal_buses if str(b) in bus_to_zoneidx_str}
print(f"Coal-eligible zones (real PUDL siting): {sorted(coal_zones)} ({len(coal_zones)} of {N_ZONES})")

# Heat-rate inputs in MMBtu/MWh.
REAL_HEAT_RATES = {"CCGT": 8.626198987641619, "OCGT": 11.840323163748854, "Coal": 10.646461985663231}

thermal_rows = []
for tech_label, atb_label, fuel_label in [("CCGT", "Gas_CC", "Gas"), ("OCGT", "Gas_CT", "Gas"), ("Coal", "Coal", "coal")]:
    row = atb.loc[atb_label]
    for zone_i in range(1, N_ZONES + 1):
        if tech_label == "Coal" and zone_i not in coal_zones:
            continue
        heat_rate = REAL_HEAT_RATES.get(tech_label, row["heat_rate_mmbtu_per_mwh"])
        thermal_rows.append({
            "Resource": f"ERCOT{zone_i}_{tech_label}",
            "Zone": zone_i,
            "Model": 1,
            "New_Build": 1,
            "Can_Retire": 1,
            "Existing_Cap_MW": 0,
            "Max_Cap_MW": -1,
            "Min_Cap_MW": 0,
            "Inv_Cost_per_MWyr": row["capex_per_kw"] * 1000 * row["annualization_rate"],
            "Fixed_OM_Cost_per_MWyr": row["opex_fixed_per_kw"] * 1000,
            "Var_OM_Cost_per_MWh": row["opex_variable_per_mwh"],
            "Heat_Rate_MMBTU_per_MWh": heat_rate,
            "Fuel": fuel_label,
            "Cap_Size": 0,
            "Start_Cost_per_MW": 0,
            "Start_Fuel_MMBTU_per_MW": 0,
            "Up_Time": 1,
            "Down_Time": 1,
            "Ramp_Up_Percentage": 1,
            "Ramp_Dn_Percentage": 1,
            "Min_Power": 0,
            "Reg_Max": 0,
            "Rsv_Max": 0,
            "Reg_Cost": 0,
            "Rsv_Cost": 0,
            "region": f"ERCOT{zone_i}",
            "cluster": 1,
        })
thermal = pd.DataFrame(thermal_rows)
thermal.to_csv(f"{OUT}/resources/Thermal.csv", index=False)
print(f"Thermal.csv: {len(thermal)} rows (CCGT + OCGT in all {N_ZONES} zones, "
      f"Coal in {len(coal_zones)} coal-siting zones only)")

### Vre.csv

Combine Step 7 wind and solar potentials with Step 9 costs.


In [ ]:
wind_pot = pd.read_csv(f"{RAW}/ercot_onwind_potential_20zone.csv", index_col=0).iloc[:, 0]
wind_pot.index = wind_pot.index.astype(str)
solar_pot = pd.read_csv(f"{RAW}/ercot_solar_potential_20zone.csv", index_col=0).iloc[:, 0]
solar_pot.index = solar_pot.index.astype(str)

vre_rows = []
for tech_label, pot_series in [("onwind", wind_pot), ("solar", solar_pot)]:
    row = atb.loc[tech_label]
    for zone_i, bus_name in enumerate(n.buses.index, start=1):
        max_cap = pot_series.get(bus_name, 0)
        vre_rows.append({
            "Resource": f"ERCOT{zone_i}_{tech_label}",
            "Zone": zone_i,
            "Num_VRE_Bins": 1,
            "New_Build": 1,
            "Can_Retire": 1,
            "Existing_Cap_MW": 0,
            "Max_Cap_MW": max_cap if max_cap > 0 else -1,
            "Min_Cap_MW": 0,
            "Inv_Cost_per_MWyr": row["capex_per_kw"] * 1000 * row["annualization_rate"],
            "Fixed_OM_Cost_per_MWyr": row["opex_fixed_per_kw"] * 1000,
            # Fill missing variable O&M with zero.
            "Var_OM_Cost_per_MWh": 0 if pd.isna(row["opex_variable_per_mwh"]) else row["opex_variable_per_mwh"],
            "Reg_Max": 0,
            "Rsv_Max": 0,
            "Reg_Cost": 0,
            "Rsv_Cost": 0,
            "region": f"ERCOT{zone_i}",
            "cluster": 1,
        })
vre = pd.DataFrame(vre_rows)
vre.to_csv(f"{OUT}/resources/Vre.csv", index=False)
print(f"Vre.csv: {len(vre)} rows")

### Storage.csv

Write battery resource rows using Step 9 power and energy costs.


In [ ]:
batt = atb.loc["battery"]
storage_rows = []
for zone_i in range(1, N_ZONES + 1):
    storage_rows.append({
        "Resource": f"ERCOT{zone_i}_battery",
        "Zone": zone_i,
        "Model": 1,
        "New_Build": 1,
        "Can_Retire": 1,
        "Existing_Cap_MW": 0,
        "Existing_Cap_MWh": 0,
        "Max_Cap_MW": -1,
        "Max_Cap_MWh": -1,
        "Min_Cap_MW": 0,
        "Min_Cap_MWh": 0,
        "Inv_Cost_per_MWyr": batt["power_cost_per_kw"] * 1000 * batt["annualization_rate"],
        "Inv_Cost_per_MWhyr": batt["energy_cost_per_kwh"] * 1000 * batt["annualization_rate"],
        "Fixed_OM_Cost_per_MWyr": batt["power_fom_per_kw"] * 1000,
        "Fixed_OM_Cost_per_MWhyr": batt["energy_fom_per_kwh"] * 1000,
        "Var_OM_Cost_per_MWh": 0,
        "Var_OM_Cost_per_MWh_In": 0,
        "Self_Disch": 0,
        "Eff_Up": 0.92,  # Equal charge/discharge efficiencies: 0.92² ≈ 0.85 round trip.
        "Eff_Down": 0.92,
        "Min_Duration": 1,
        "Max_Duration": 8,
        "Reg_Max": 1.0,
        "Rsv_Max": 1.0,
        "Reg_Cost": 0,
        "Rsv_Cost": 0,
        "region": f"ERCOT{zone_i}",
        "cluster": 1,
        "LDS": 1,
    })
storage = pd.DataFrame(storage_rows)
storage.to_csv(f"{OUT}/resources/Storage.csv", index=False)
print(f"Storage.csv: {len(storage)} rows")

### Generators_variability.csv

Export Step 7 wind and solar profiles and the availability columns defined below.


In [ ]:
wind_prof = pd.read_csv(f"{RAW}/ercot_onwind_profile_20zone.csv", index_col=0)
solar_prof = pd.read_csv(f"{RAW}/ercot_solar_profile_20zone.csv", index_col=0)

gen_var_cols = {"Time_Index": np.arange(1, 8761)}
for zone_i, bus_name in enumerate(n.buses.index, start=1):
    col_name = f"z{zone_i}"
    gen_var_cols[f"ERCOT{zone_i}_onwind"] = wind_prof[col_name].to_numpy() if col_name in wind_prof.columns else 0.0
    gen_var_cols[f"ERCOT{zone_i}_solar"] = solar_prof[col_name].to_numpy() if col_name in solar_prof.columns else 0.0
    gen_var_cols[f"ERCOT{zone_i}_battery"] = 1.0
    for tech in ["CCGT", "OCGT", "Coal"]:
        if tech == "Coal" and zone_i not in coal_zones:
            continue
        gen_var_cols[f"ERCOT{zone_i}_{tech}"] = 1.0

gen_var = pd.DataFrame(gen_var_cols)
gen_var.to_csv(f"{OUT}/system/Generators_variability.csv", index=False)
print(f"Generators_variability.csv: {gen_var.shape}")

print(f"\n=== Assembly complete. Output: {OUT} ===")

## Check exported files

Run the checks below before proceeding to Step 12.


In [ ]:
# Summarize exported line capacities and parallel-circuit counts.
net_check = pd.read_csv(f"{OUT}/system/Network.csv")
flows = net_check["Line_Max_Flow_MW"].dropna()
print(f"Line_Max_Flow_MW: n={len(flows)}, min={flows.min():.0f}, median={flows.median():.0f}, "
      f"max={flows.max():.0f} MW")
print(f"Line_Num_Parallel: min={net_check['Line_Num_Parallel'].min():.2f}, "
      f"max={net_check['Line_Num_Parallel'].max():.2f}")

In [ ]:
# Summarize exported demand: peak, mean and annual energy.
demand_check = pd.read_csv(f"{OUT}/system/Demand_data.csv")
demand_cols = [c for c in demand_check.columns if c.startswith("Demand_MW_z")]
total_check = demand_check.loc[1:, demand_cols].astype(float).sum(axis=1)
print(f"System peak: {total_check.max():,.0f} MW | average: {total_check.mean():,.0f} MW | "
      f"annual energy: {total_check.sum() / 1e6:.1f} TWh")

In [ ]:
# Summarize exported wind and solar capacity factors.
gv_check = pd.read_csv(f"{OUT}/system/Generators_variability.csv")
wind_cols = [c for c in gv_check.columns if c.endswith("_onwind")]
solar_cols = [c for c in gv_check.columns if c.endswith("_solar")]
print(f"Wind zone-mean CF: {gv_check[wind_cols].mean().mean():.3f} (ERCOT ELCC study: 0.389)")
print(f"Solar zone-mean CF (all-hours): {gv_check[solar_cols].mean().mean():.3f} (ERCOT ELCC study: 0.265, "
      f"daytime-only will read higher)")

## Step 12 — Write per-zone thermal caps

Run `build_thermal_zonal_cap.py` to write `resources_thermal_zonal_cap/` and
`policies_thermal_zonal_cap/` in the case directory. The default cap is 5000 MW per zone.
To generate another cap, pass its value in MW, for example:
`run("build_thermal_zonal_cap.py", CASE_BASELINE, 3000)`.


In [ ]:
run("build_thermal_zonal_cap.py", CASE_BASELINE)
